In [2]:
from google.colab import drive
import os

# Drive'ı bağla
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')
    print("✅ Drive tekrar bağlandı.")

# Proje klasörünü kontrol et
proje_yolu = '/content/drive/MyDrive/Embedded'
if os.path.exists(proje_yolu):
    print(f"📂 Proje klasörü yerinde duruyor: {proje_yolu}")
else:
    print("❌ Klasör yolu bulunamadı, Drive bağlantısını kontrol et.")

📂 Proje klasörü yerinde duruyor: /content/drive/MyDrive/Embedded


In [4]:
import xml.etree.ElementTree as ET
import os

classes = ["object"]  # TEK SINIF, adı önemli değil

xml_folder = "/content/drive/MyDrive/Embedded/dataset/annotations"
img_folder = "/content/drive/MyDrive/Embedded/dataset/image"
label_folder = "/content/drive/MyDrive/Embedded/dataset/label"

os.makedirs(label_folder, exist_ok=True)

def convert_bbox(size, box):
    dw = 1. / size[0]
    dh = 1. / size[1]
    x = (box[0] + box[1]) / 2.0
    y = (box[2] + box[3]) / 2.0
    w = box[1] - box[0]
    h = box[3] - box[2]
    return (x * dw, y * dh, w * dw, h * dh)

for xml_file in os.listdir(xml_folder):
    if not xml_file.endswith(".xml"):
        continue

    tree = ET.parse(os.path.join(xml_folder, xml_file))
    root = tree.getroot()

    size = root.find("size")
    w = int(size.find("width").text)
    h = int(size.find("height").text)

    txt_name = xml_file.replace(".xml", ".txt")
    txt_path = os.path.join(label_folder, txt_name)

    with open(txt_path, "w") as f:
        for obj in root.findall("object"):
            cls_name = obj.find("name").text
            cls_id = 0  # tek sınıf

            xml_box = obj.find("bndbox")
            bbox = (
                float(xml_box.find("xmin").text),
                float(xml_box.find("xmax").text),
                float(xml_box.find("ymin").text),
                float(xml_box.find("ymax").text)
            )

            bb = convert_bbox((w, h), bbox)
            f.write(f"{cls_id} {' '.join(map(str, bb))}\n")


ParseError: no element found: line 1, column 0 (<string>)

In [9]:
import os
import shutil
import random
import yaml

# ==========================================
# 1. AYARLAR (Burayı Kontrol Edin)
# ==========================================
# Drive'daki dataset ana klasörünüzün yolu
base_dir = '/content/drive/MyDrive/Embedded/dataset'

# Kaynak Klasörler (Şu an resimlerin ve txt'lerin olduğu yerler)
images_source_dir = os.path.join(base_dir, 'images')
labels_source_dir = os.path.join(base_dir, 'labels')

# Hedef Klasör Yapısı (YOLO'nun istediği yapı)
# images/train, images/val, labels/train, labels/val
img_train_dir = os.path.join(base_dir, 'images', 'train')
img_val_dir = os.path.join(base_dir, 'images', 'val')
lbl_train_dir = os.path.join(base_dir, 'labels', 'train')
lbl_val_dir = os.path.join(base_dir, 'labels', 'val')

# Ayırma Oranı (%80 Eğitim, %20 Test)
split_ratio = 0.8

# ==========================================
# 2. KLASÖRLERİ OLUŞTURMA
# ==========================================
def create_dir(path):
    if not os.path.exists(path):
        os.makedirs(path)

# Hedef klasörleri oluştur
create_dir(img_train_dir)
create_dir(img_val_dir)
create_dir(lbl_train_dir)
create_dir(lbl_val_dir)

# ==========================================
# 3. DOSYALARI TAŞIMA VE AYIRMA
# ==========================================
# Sadece kök dizindeki dosyaları al (alt klasördekileri alma)
all_images = [f for f in os.listdir(images_source_dir)
              if os.path.isfile(os.path.join(images_source_dir, f))
              and f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]

# Rastgele karıştır
random.shuffle(all_images)

# İndeksi hesapla
train_idx = int(len(all_images) * split_ratio)

# Dosyaları taşıma fonksiyonu
def move_files(file_list, destination_img_dir, destination_lbl_dir):
    count = 0
    for img_name in file_list:
        # Resim dosyasının tam yolları
        src_img_path = os.path.join(images_source_dir, img_name)
        dst_img_path = os.path.join(destination_img_dir, img_name)

        # Etiket dosyası ismini bul (.jpg -> .txt)
        label_name = os.path.splitext(img_name)[0] + '.txt'
        src_label_path = os.path.join(labels_source_dir, label_name)
        dst_label_path = os.path.join(destination_lbl_dir, label_name)

        # Eğer etiket dosyası varsa taşı
        if os.path.exists(src_label_path):
            shutil.move(src_img_path, dst_img_path)
            shutil.move(src_label_path, dst_label_path)
            count += 1
        else:
            print(f"Uyarı: {img_name} için etiket dosyası bulunamadı, taşınmadı.")
    return count

print("Dosyalar taşınıyor...")
train_count = move_files(all_images[:train_idx], img_train_dir, lbl_train_dir)
val_count = move_files(all_images[train_idx:], img_val_dir, lbl_val_dir)

print(f"\n✅ İŞLEM TAMAMLANDI!")
print(f"Eğitim Seti (Train): {train_count} resim/etiket")
print(f"Test Seti (Val):   {val_count} resim/etiket")

# ==========================================
# 4. DATA.YAML DOSYASINI OLUŞTURMA
# ==========================================
data_config = {
    'path': base_dir,
    'train': 'images/train',
    'val': 'images/val',
    'nc': 10,
    'names': ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9']
}

yaml_file_path = os.path.join(base_dir, 'data.yaml')
with open(yaml_file_path, 'w') as f:
    yaml.dump(data_config, f, default_flow_style=False, sort_keys=False)

print(f"\n📄 data.yaml dosyası da güncellendi: {yaml_file_path}")

Dosyalar taşınıyor...
Uyarı: IMG_0091.jpg için etiket dosyası bulunamadı, taşınmadı.
Uyarı: IMG_0085.jpg için etiket dosyası bulunamadı, taşınmadı.
Uyarı: IMG_0037.jpg için etiket dosyası bulunamadı, taşınmadı.
Uyarı: IMG_0033.jpg için etiket dosyası bulunamadı, taşınmadı.
Uyarı: IMG_0046.jpg için etiket dosyası bulunamadı, taşınmadı.
Uyarı: IMG_0084.jpg için etiket dosyası bulunamadı, taşınmadı.
Uyarı: IMG_0035.jpg için etiket dosyası bulunamadı, taşınmadı.
Uyarı: IMG_0023.jpg için etiket dosyası bulunamadı, taşınmadı.
Uyarı: IMG_0027.jpg için etiket dosyası bulunamadı, taşınmadı.
Uyarı: IMG_0052.jpg için etiket dosyası bulunamadı, taşınmadı.
Uyarı: IMG_0079.jpg için etiket dosyası bulunamadı, taşınmadı.

✅ İŞLEM TAMAMLANDI!
Eğitim Seti (Train): 45 resim/etiket
Test Seti (Val):   14 resim/etiket

📄 data.yaml dosyası da güncellendi: /content/drive/MyDrive/Embedded/dataset/data.yaml


In [10]:
# ==========================================
# 1. KURULUM (Ultralytics YOLO)
# ==========================================
# Eğer kurulu değilse kur
%pip install ultralytics
import ultralytics
from ultralytics import YOLO
import os

print(f"Ultralytics Version: {ultralytics.__version__}")

# ==========================================
# 2. EĞİTİM AYARLARI
# ==========================================
# Veri seti ayar dosyasının tam yolu (Az önce oluşturduğumuz)
data_yaml_path = '/content/drive/MyDrive/Embedded/dataset/data.yaml'

# Modeli Yükle (Nano versiyon - Gömülü sistemler için en iyisi)
model = YOLO('yolov8n.pt')

# ==========================================
# 3. EĞİTİMİ BAŞLAT
# ==========================================
# epochs: 50 (Veri az olduğu için çabuk biter, duruma göre artırabilirsin)
# imgsz: 320 (ESP32 kameraları genelde düşük çözünürlük kullanır, 320x320 idealdir)
print("🚀 Eğitim Başlıyor...")

results = model.train(
    data=data_yaml_path,
    epochs=100,      # 100 Epoch yapalım, veri az olduğu için ezberlemesi (overfit) gerekebilir
    imgsz=320,       # Gömülü sistem dostu boyut
    batch=16,
    name='yolo_esp32_project', # Proje ismi
    project='/content/drive/MyDrive/Embedded/training_results' # Sonuçları Drive'a kaydet
)

print("✅ Eğitim Tamamlandı!")

# ==========================================
# 4. MODELİ TFLITE FORMATINA ÇEVİRME (ESP32 İÇİN)
# ==========================================
print("🔄 Model TFLite formatına çevriliyor (ESP32 için)...")

# En iyi modeli al
best_model_path = '/content/drive/MyDrive/Embedded/training_results/yolo_esp32_project/weights/best.pt'
model = YOLO(best_model_path)

# TFLite'a export et (int8 quantization ile daha da küçültülebilir ama şimdilik standart fp32/fp16 yapalım)
model.export(format='tflite', imgsz=320)

print(f"\n📂 TFLite modelin şurada hazır: {best_model_path.replace('.pt', '.tflite')}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 32.0 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics Version: 8.4.3
🚀 Eğitim Başlıyor...
Ultralytics 8.4.3 🚀 Python-3.12.12 torch-2.9.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/Embedded/dataset/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, f

KeyboardInterrupt: 

In [11]:
import xml.etree.ElementTree as ET
import os

# --- AYARLAR ---
xml_folder = "/content/drive/MyDrive/Embedded/dataset/annotations"
label_folder = "/content/drive/MyDrive/Embedded/dataset/label"  # Burası 'labels' olmalı aslında YOLO için

os.makedirs(label_folder, exist_ok=True)

def convert_bbox(size, box):
    dw = 1. / size[0]
    dh = 1. / size[1]
    x = (box[0] + box[1]) / 2.0
    y = (box[2] + box[3]) / 2.0
    w = box[1] - box[0]
    h = box[3] - box[2]
    return (x * dw, y * dh, w * dw, h * dh)

print("🚀 Dönüştürme işlemi başlıyor...")
success_count = 0
error_count = 0

for xml_file in os.listdir(xml_folder):
    if not xml_file.endswith(".xml"):
        continue

    file_path = os.path.join(xml_folder, xml_file)

    try:
        # XML dosyasını okumaya çalış
        tree = ET.parse(file_path)
        root = tree.getroot()

        # Resim boyutlarını al
        size = root.find("size")
        if size is None: # Bazen size etiketi eksik olabilir
            print(f"⚠️ Hata: {xml_file} içinde <size> etiketi yok, atlanıyor.")
            continue

        w = int(size.find("width").text)
        h = int(size.find("height").text)

        # .txt dosyasını hazırla
        txt_name = xml_file.replace(".xml", ".txt")
        txt_path = os.path.join(label_folder, txt_name)

        with open(txt_path, "w") as f:
            for obj in root.findall("object"):
                cls_name = obj.find("name").text

                # --- KRİTİK DÜZELTME: Sınıf ID'sini dinamik al ---
                # Eğer etiketlerin "0", "1", "9" şeklindeyse direkt sayıya çevir
                try:
                    cls_id = int(cls_name)
                except ValueError:
                    # Eğer sınıf ismin sayı değilse (örn: "cat"), bunu bir harita ile çözmelisin
                    # Şimdilik hata vermesin diye 0 yapıyoruz ama burayı kontrol et
                    print(f"⚠️ Uyarı: {xml_file} içindeki sınıf ismi '{cls_name}' sayıya çevrilemedi. 0 atandı.")
                    cls_id = 0

                xml_box = obj.find("bndbox")
                bbox = (
                    float(xml_box.find("xmin").text),
                    float(xml_box.find("xmax").text),
                    float(xml_box.find("ymin").text),
                    float(xml_box.find("ymax").text)
                )

                bb = convert_bbox((w, h), bbox)
                f.write(f"{cls_id} {' '.join(map(str, bb))}\n")

        success_count += 1

    except ET.ParseError:
        print(f"❌ BOZUK DOSYA: {xml_file} (İçi boş veya hatalı XML) -> Atlandı.")
        error_count += 1
    except Exception as e:
        print(f"❌ Beklenmedik Hata: {xml_file} -> {e}")
        error_count += 1

print("-" * 30)
print(f"✅ Başarılı: {success_count} dosya")
print(f"❌ Hatalı:   {error_count} dosya")
print(f"📂 Etiketler şuraya kaydedildi: {label_folder}")

🚀 Dönüştürme işlemi başlıyor...
❌ BOZUK DOSYA: IMG_0079.xml (İçi boş veya hatalı XML) -> Atlandı.
------------------------------
✅ Başarılı: 69 dosya
❌ Hatalı:   1 dosya
📂 Etiketler şuraya kaydedildi: /content/drive/MyDrive/Embedded/dataset/label


In [12]:
import os
import shutil
import random
import yaml

# ==========================================
# 1. AYARLAR (Kendi yollarını kontrol et)
# ==========================================
base_dir = '/content/drive/MyDrive/Embedded/dataset'

# Kaynaklar (Resimlerin ve YENİ etiketlerin olduğu yerler)
# NOT: Eğer resimlerin dağınıksa onları önce 'image' klasörüne toplaman iyi olur.
source_images = os.path.join(base_dir, 'image')  # Senin XML kodunda 'image' yazıyordu
source_labels = os.path.join(base_dir, 'label')  # Yeni oluşturduğun .txt'ler burada

# Hedefler (YOLO'nun okuyacağı yerler)
train_img_dir = os.path.join(base_dir, 'images', 'train')
val_img_dir = os.path.join(base_dir, 'images', 'val')
train_lbl_dir = os.path.join(base_dir, 'labels', 'train')
val_lbl_dir = os.path.join(base_dir, 'labels', 'val')

# ==========================================
# 2. TEMİZLİK (Eski hatalı dağıtımı sil)
# ==========================================
print("🧹 Eski train/val klasörleri temizleniyor...")
for folder in [train_img_dir, val_img_dir, train_lbl_dir, val_lbl_dir]:
    if os.path.exists(folder):
        shutil.rmtree(folder)  # Klasörü içindekilerle sil
    os.makedirs(folder, exist_ok=True) # Yeniden boş oluştur

# ==========================================
# 3. EŞLEŞTİRME VE TAŞIMA
# ==========================================
# Sadece etiketi olan resimleri bul (Böylece etiketsiz resim hatası almazsın)
valid_files = []

# Tüm etiket dosyalarını listele
all_labels = [f for f in os.listdir(source_labels) if f.endswith('.txt')]

print(f"🔍 Toplam {len(all_labels)} adet etiket dosyası bulundu.")

for lbl_file in all_labels:
    # Etiketin ismine karşılık gelen resmi bul (jpg, png, jpeg...)
    base_name = os.path.splitext(lbl_file)[0]

    found_image = None
    for ext in ['.jpg', '.jpeg', '.png', '.bmp']:
        img_name = base_name + ext
        img_path = os.path.join(source_images, img_name)
        if os.path.exists(img_path):
            found_image = img_name
            break

    if found_image:
        valid_files.append({'image': found_image, 'label': lbl_file})
    else:
        print(f"⚠️ Uyarı: {lbl_file} için resim dosyası bulunamadı!")

# Karıştır
random.shuffle(valid_files)

# Ayır (%80 Train, %20 Val)
split_idx = int(len(valid_files) * 0.8)
train_set = valid_files[:split_idx]
val_set = valid_files[split_idx:]

print(f"📦 Dosyalar kopyalanıyor... (Train: {len(train_set)}, Val: {len(val_set)})")

def copy_data(file_list, img_dest, lbl_dest):
    for item in file_list:
        # Kopyala (Move değil Copy yapıyoruz, orijinaller kalsın garanti olsun)
        shutil.copy(os.path.join(source_images, item['image']), os.path.join(img_dest, item['image']))
        shutil.copy(os.path.join(source_labels, item['label']), os.path.join(lbl_dest, item['label']))

copy_data(train_set, train_img_dir, train_lbl_dir)
copy_data(val_set, val_img_dir, val_lbl_dir)

# ==========================================
# 4. DATA.YAML GÜNCELLEME
# ==========================================
data_config = {
    'path': base_dir,
    'train': 'images/train',
    'val': 'images/val',
    'nc': 10,  # 0-9 Rakamlar
    'names': ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9']
}

with open(os.path.join(base_dir, 'data.yaml'), 'w') as f:
    yaml.dump(data_config, f, default_flow_style=False, sort_keys=False)

print("\n✅ HAZIR! Artık modeli çalıştırabilirsin.")

🧹 Eski train/val klasörleri temizleniyor...
🔍 Toplam 69 adet etiket dosyası bulundu.
⚠️ Uyarı: IMG_0086.txt için resim dosyası bulunamadı!
⚠️ Uyarı: IMG_0092.txt için resim dosyası bulunamadı!
⚠️ Uyarı: IMG_0070.txt için resim dosyası bulunamadı!
⚠️ Uyarı: IMG_0073.txt için resim dosyası bulunamadı!
⚠️ Uyarı: IMG_0072.txt için resim dosyası bulunamadı!
⚠️ Uyarı: IMG_0058.txt için resim dosyası bulunamadı!
⚠️ Uyarı: IMG_0064.txt için resim dosyası bulunamadı!
⚠️ Uyarı: IMG_0066.txt için resim dosyası bulunamadı!
⚠️ Uyarı: IMG_0067.txt için resim dosyası bulunamadı!
⚠️ Uyarı: IMG_0051.txt için resim dosyası bulunamadı!
⚠️ Uyarı: IMG_0078.txt için resim dosyası bulunamadı!
⚠️ Uyarı: IMG_0071.txt için resim dosyası bulunamadı!
⚠️ Uyarı: IMG_0059.txt için resim dosyası bulunamadı!
⚠️ Uyarı: IMG_0061.txt için resim dosyası bulunamadı!
⚠️ Uyarı: IMG_0040.txt için resim dosyası bulunamadı!
⚠️ Uyarı: IMG_0075.txt için resim dosyası bulunamadı!
⚠️ Uyarı: IMG_0044.txt için resim dosyası bulunamad

In [13]:
import xml.etree.ElementTree as ET
import os

# --- AYARLAR ---
# Klasör yollarını senin Drive yapına göre ayarladım
base_dir = '/content/drive/MyDrive/Embedded/dataset'
xml_folder = os.path.join(base_dir, 'annotations')
label_folder = os.path.join(base_dir, 'label')  # Geçici label klasörü

os.makedirs(label_folder, exist_ok=True)

def convert_bbox(size, box):
    dw = 1. / size[0]
    dh = 1. / size[1]
    x = (box[0] + box[1]) / 2.0
    y = (box[2] + box[3]) / 2.0
    w = box[1] - box[0]
    h = box[3] - box[2]
    return (x * dw, y * dh, w * dw, h * dh)

print("🚀 XML -> TXT Dönüşümü Başlıyor...")
success = 0
errors = 0

if not os.path.exists(xml_folder):
    print(f"❌ HATA: '{xml_folder}' klasörü bulunamadı! Lütfen XML dosyalarını yüklediğinden emin ol.")
else:
    for xml_file in os.listdir(xml_folder):
        if not xml_file.endswith(".xml"): continue

        try:
            tree = ET.parse(os.path.join(xml_folder, xml_file))
            root = tree.getroot()

            # Resim boyutunu al
            size = root.find("size")
            w = int(size.find("width").text)
            h = int(size.find("height").text)

            # TXT dosyasını oluştur
            txt_path = os.path.join(label_folder, xml_file.replace(".xml", ".txt"))

            with open(txt_path, "w") as f:
                for obj in root.findall("object"):
                    # Sınıf ismini al (0, 1, 2... gibi sayı olmalı)
                    name = obj.find("name").text
                    try:
                        cls_id = int(name) # Sayıysa direkt al
                    except:
                        cls_id = 0 # Sayı değilse (örn: 'car') 0 yap

                    xml_box = obj.find("bndbox")
                    b = (float(xml_box.find("xmin").text), float(xml_box.find("xmax").text),
                         float(xml_box.find("ymin").text), float(xml_box.find("ymax").text))

                    # YOLO formatına çevir
                    bb = convert_bbox((w, h), b)
                    f.write(f"{cls_id} {' '.join(map(str, bb))}\n")
            success += 1
        except Exception as e:
            print(f"⚠️ Hata ({xml_file}): {e}")
            errors += 1

    print(f"\n✅ DÖNÜŞÜM BİTTİ: {success} dosya oluşturuldu. (Hata: {errors})")

🚀 XML -> TXT Dönüşümü Başlıyor...
⚠️ Hata (IMG_0079.xml): no element found: line 1, column 0

✅ DÖNÜŞÜM BİTTİ: 69 dosya oluşturuldu. (Hata: 1)


In [14]:
import os
import shutil
import random
import yaml

# --- AYARLAR ---
base_dir = '/content/drive/MyDrive/Embedded/dataset'
source_images = os.path.join(base_dir, 'images') # Senin yüklediğin resimler
source_labels = os.path.join(base_dir, 'label')  # Az önce oluşturduğumuz txt'ler

# Hedef Klasörler (YOLO burayı kullanacak)
train_img_dir = os.path.join(base_dir, 'images', 'train')
val_img_dir = os.path.join(base_dir, 'images', 'val')
train_lbl_dir = os.path.join(base_dir, 'labels', 'train')
val_lbl_dir = os.path.join(base_dir, 'labels', 'val')

print("🧹 Eski eğitim klasörleri temizleniyor ve yeniden oluşturuluyor...")
# Temizlik
for d in [train_img_dir, val_img_dir, train_lbl_dir, val_lbl_dir]:
    if os.path.exists(d): shutil.rmtree(d)
    os.makedirs(d, exist_ok=True)

# Eşleştirme
pairs = []
if not os.path.exists(source_images):
    print(f"❌ HATA: '{source_images}' klasörü yok! Resimleri yükledin mi?")
else:
    all_labels = os.listdir(source_labels)
    print(f"🔍 {len(all_labels)} etiket taranıyor...")

    for lbl in all_labels:
        if not lbl.endswith('.txt'): continue
        base_name = lbl[:-4] # uzantıyı at

        # Resmi bul (jpg, png vs.)
        img_name = None
        for ext in ['.jpg', '.jpeg', '.png', '.JPG']:
            if os.path.exists(os.path.join(source_images, base_name + ext)):
                img_name = base_name + ext
                break

        if img_name:
            pairs.append((img_name, lbl))

    if len(pairs) == 0:
        print("❌ HATA: Hiçbir resim ve etiket eşleşmedi! İsimlerin aynı olduğundan emin ol.")
    else:
        # Karıştır ve Dağıt
        random.shuffle(pairs)
        split = int(len(pairs) * 0.8)
        train_set = pairs[:split]
        val_set = pairs[split:]

        print(f"📦 Dosyalar taşınıyor... (Eğitim: {len(train_set)}, Test: {len(val_set)})")

        def copy_files(file_list, img_dst, lbl_dst):
            for img, lbl in file_list:
                shutil.copy(os.path.join(source_images, img), os.path.join(img_dst, img))
                shutil.copy(os.path.join(source_labels, lbl), os.path.join(lbl_dst, lbl))

        copy_files(train_set, train_img_dir, train_lbl_dir)
        copy_files(val_set, val_img_dir, val_lbl_dir)

        # data.yaml oluştur
        data = {
            'path': base_dir,
            'train': 'images/train',
            'val': 'images/val',
            'nc': 10,
            'names': ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9']
        }
        with open(os.path.join(base_dir, 'data.yaml'), 'w') as f:
            yaml.dump(data, f, sort_keys=False)

        print("\n✅ HAZIRLIK TAMAMLANDI! Modele geçebilirsin.")

🧹 Eski eğitim klasörleri temizleniyor ve yeniden oluşturuluyor...
🔍 69 etiket taranıyor...
📦 Dosyalar taşınıyor... (Eğitim: 55, Test: 14)

✅ HAZIRLIK TAMAMLANDI! Modele geçebilirsin.
